# Synthetic Structure Recovery Benchmark

Reproduces the synthetic structure recovery experiment from Mansinghka et al. (2016) Figure 7.

Generates data with known structure (2 views, 3 clusters per view), runs CrossCat
collapsed Gibbs inference, and measures recovery via Adjusted Rand Index (ARI)
and dependence probability matrix.

**Pass criteria:**
- Column partition ARI >= 0.80
- Row clustering ARI >= 0.70 (each view)
- Within-view dependence probability >= 0.80
- Between-view dependence probability <= 0.20

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    # Checkout branch: try local first, then create from remote tracking branch
    checkout = subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR)
    if checkout.returncode != 0:
        subprocess.run(
            ["git", "checkout", "-b", BRANCH, f"origin/{BRANCH}"], cwd=WORKDIR, check=True
        )
    else:
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from benchmarks.utils import (
    create_results_dir,
    detect_platform,
    plot_convergence,
    plot_z_matrix,
    save_metrics,
)
from crosscat import (
    ColumnType,
    adjusted_rand_index,
    collect_diagnostics,
    column_partition_ari,
    initialize,
    pack_state,
    packed_dependence_matrix,
    packed_gibbs_sweep,
    unpack_state,
)

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Configuration

In [ ]:
N_ROWS = 200
N_SWEEPS = 200
N_CHAINS = 10
DIAG_INTERVAL = 5
SEED = 42

## 3. Generate Synthetic Data

2 views x 4 columns each, 3 well-separated Gaussian clusters per view.

In [ ]:
rng_key = jax.random.key(SEED)
k_data, k_init = jax.random.split(rng_key)
k1, k2, k3, k4 = jax.random.split(k_data, 4)

# View 0: 3 clusters, columns 0-3
cluster_assignments_v0 = jax.random.categorical(
    k1, jnp.log(jnp.array([1 / 3, 1 / 3, 1 / 3])), shape=(N_ROWS,)
)
means_v0 = jnp.array([[-3.0] * 4, [0.0] * 4, [3.0] * 4])
data_v0 = means_v0[cluster_assignments_v0] + jax.random.normal(k2, shape=(N_ROWS, 4)) * 0.5

# View 1: 3 clusters, columns 4-7
cluster_assignments_v1 = jax.random.categorical(
    k3, jnp.log(jnp.array([1 / 3, 1 / 3, 1 / 3])), shape=(N_ROWS,)
)
means_v1 = jnp.array([[-4.0] * 4, [0.0] * 4, [4.0] * 4])
data_v1 = means_v1[cluster_assignments_v1] + jax.random.normal(k4, shape=(N_ROWS, 4)) * 0.5

data = jnp.concatenate([data_v0, data_v1], axis=1)
col_types = [ColumnType.CONTINUOUS] * 8
true_col_assign = jnp.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=jnp.int32)
true_row_assign = [cluster_assignments_v0, cluster_assignments_v1]

print(f"Data: {data.shape}, 2 views x 4 cols, 3 clusters each")

## 4. Run Gibbs Inference

Multi-chain collapsed Gibbs sampling with batched sweeps for GPU throughput.

In [ ]:
def best_view_match(state, true_row_assignments):
    """Find best ARI match between inferred and true views."""
    best_aris = []
    for true_assign in true_row_assignments:
        best_ari = -1.0
        for view in state.views:
            ari = float(adjusted_rand_index(true_assign, view.row_assignments))
            best_ari = max(best_ari, ari)
        best_aris.append(best_ari)
    return best_aris[0], best_aris[1]


init_keys = jax.random.split(k_init, N_CHAINS)
states = []
all_chain_metrics = []

for chain_idx in range(N_CHAINS):
    print(f"\n--- Chain {chain_idx + 1}/{N_CHAINS} ---")
    k_i, k_sweep = jax.random.split(init_keys[chain_idx])
    state = initialize(k_i, data, col_types).state
    packed = pack_state(state)
    chain_metrics = []
    t0 = time.time()

    sweep = 0
    while sweep < N_SWEEPS:
        batch = min(DIAG_INTERVAL, N_SWEEPS - sweep)
        k_sweep, subkey = jax.random.split(k_sweep)
        packed = packed_gibbs_sweep(subkey, packed, data, n_sweeps=batch)
        sweep += batch
        state = unpack_state(packed, col_types, data=data)
        diag = collect_diagnostics(state, data)
        col_ari = float(column_partition_ari(state, true_col_assign))
        row_ari_v0, row_ari_v1 = best_view_match(state, true_row_assign)
        chain_metrics.append(
            {
                "sweep": sweep,
                "col_ari": col_ari,
                "row_ari_v0": row_ari_v0,
                "row_ari_v1": row_ari_v1,
                **diag,
            }
        )
        del state
        if sweep % 10 == 0 or sweep == N_SWEEPS:
            print(f"  Sweep {sweep:3d}/{N_SWEEPS}: col_ARI={col_ari:.3f}")

    state = unpack_state(packed, col_types, data=data)
    elapsed = time.time() - t0
    print(f"  Time: {elapsed:.1f}s ({elapsed / N_SWEEPS:.2f}s/sweep)")
    states.append(state)
    all_chain_metrics.append(chain_metrics)

## 5. Recovery Metrics

In [ ]:
print("=" * 60)
print("RECOVERY METRICS")
print("=" * 60)

results = {}

# Column partition ARI
col_aris = [float(column_partition_ari(s, true_col_assign)) for s in states]
mean_col_ari = sum(col_aris) / len(col_aris)
results["col_ari_mean"] = mean_col_ari

# Row clustering ARI
row_aris_v0, row_aris_v1 = [], []
for s in states:
    v0, v1 = best_view_match(s, true_row_assign)
    row_aris_v0.append(v0)
    row_aris_v1.append(v1)
mean_row_ari_v0 = sum(row_aris_v0) / len(row_aris_v0)
mean_row_ari_v1 = sum(row_aris_v1) / len(row_aris_v1)
results["row_ari_v0_mean"] = mean_row_ari_v0
results["row_ari_v1_mean"] = mean_row_ari_v1

# Dependence matrix
packed_states = [pack_state(s) for s in states]
z_matrix = packed_dependence_matrix(packed_states)
within_view_prob = float((z_matrix[:4, :4].sum() + z_matrix[4:, 4:].sum() - 8.0) / (2 * (4 * 3)))
between_view_prob = float(z_matrix[:4, 4:].mean())
results["within_view_dep_prob"] = within_view_prob
results["between_view_dep_prob"] = between_view_prob


def check(name, value, threshold, higher=True):
    passed = value >= threshold if higher else value <= threshold
    status = "PASS" if passed else "FAIL"
    op = ">=" if higher else "<="
    print(f"  [{status}] {name}: {value:.3f} (threshold {op} {threshold:.2f})")
    return passed


print()
checks = [
    check("Column partition ARI (mean)", mean_col_ari, 0.80),
    check("Row clustering ARI view 0 (mean)", mean_row_ari_v0, 0.70),
    check("Row clustering ARI view 1 (mean)", mean_row_ari_v1, 0.70),
    check("Within-view dependence prob", within_view_prob, 0.80),
    check("Between-view dependence prob", between_view_prob, 0.20, higher=False),
]
results["all_passed"] = all(checks)
print(f"\n{'ALL CHECKS PASSED' if results['all_passed'] else 'SOME CHECKS FAILED'}")

## 6. Convergence Plot

In [ ]:
def average_chain_metrics(all_chain_metrics):
    n_points = len(all_chain_metrics[0])
    avg = []
    for i in range(n_points):
        combined = {"sweep": all_chain_metrics[0][i]["sweep"]}
        keys = [k for k in all_chain_metrics[0][i] if k != "sweep"]
        for key in keys:
            vals = [
                c[i].get(key) for c in all_chain_metrics if isinstance(c[i].get(key), (int, float))
            ]
            if vals:
                combined[key] = sum(vals) / len(vals)
        avg.append(combined)
    return avg


results_dir = create_results_dir("synthetic")
avg_metrics = average_chain_metrics(all_chain_metrics)
plot_convergence(avg_metrics, results_dir, ari_keys=["col_ari", "row_ari_v0", "row_ari_v1"])

from IPython.display import Image, display

display(Image(filename=str(results_dir / "convergence.png")))

## 7. Dependence Matrix (Z-matrix)

In [ ]:
plot_z_matrix(z_matrix, results_dir, col_labels=[f"col_{i}" for i in range(8)])
display(Image(filename=str(results_dir / "z_matrix.png")))

## 8. Cluster Recovery

In [ ]:
data_np = np.array(data)
colors_map = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
final_state = states[-1]

# Find best-matching views
best_view_idx = []
for true_assign in true_row_assign:
    best_ari, best_idx = -1.0, 0
    for v_idx, view in enumerate(final_state.views):
        ari = float(adjusted_rand_index(true_assign, view.row_assignments))
        if ari > best_ari:
            best_ari, best_idx = ari, v_idx
    best_view_idx.append(best_idx)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
view_col_pairs = [(0, 1), (4, 5)]
view_labels = ["View 0 (cols 0-3)", "View 1 (cols 4-7)"]

for v, (cx, cy) in enumerate(view_col_pairs):
    true_a = np.array(true_row_assign[v])
    inferred_a = np.array(final_state.views[best_view_idx[v]].row_assignments)
    for cid in range(int(true_a.max()) + 1):
        mask = true_a == cid
        axes[0, v].scatter(
            data_np[mask, cx],
            data_np[mask, cy],
            c=colors_map[cid % len(colors_map)],
            s=15,
            alpha=0.6,
            label=f"Cluster {cid}",
        )
    for cid in range(int(inferred_a.max()) + 1):
        mask = inferred_a == cid
        axes[1, v].scatter(
            data_np[mask, cx],
            data_np[mask, cy],
            c=colors_map[cid % len(colors_map)],
            s=15,
            alpha=0.6,
            label=f"Cluster {cid}",
        )
    axes[0, v].set_title(f"True \u2014 {view_labels[v]}")
    axes[1, v].set_title(f"Inferred \u2014 {view_labels[v]}")
    for row in range(2):
        axes[row, v].set_xlabel(f"Column {cx}")
        axes[row, v].set_ylabel(f"Column {cy}")
        axes[row, v].legend(fontsize=8, markerscale=2)

fig.suptitle("Cluster Recovery: True vs Inferred", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(results_dir / "cluster_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Save Results

In [ ]:
import shutil

save_metrics(
    {
        **results,
        "config": {
            "n_rows": N_ROWS,
            "n_sweeps": N_SWEEPS,
            "n_chains": N_CHAINS,
            "seed": SEED,
        },
        "per_sweep": avg_metrics,
    },
    results_dir,
)

print(f"Results saved to {results_dir}/")
archive = Path("benchmarks/results/synthetic_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")